# Full Dataset Evaluation — 4-bit (Paper Mode)

**Methods:** ICL1D+ (paper baseline) · CFR · CAFD-LC · SDCP  
**Datasets:** TruthfulQA · MMLU · ARC-Challenge  
**Model:** Mistral-7B-Instruct-v0.2 (4-bit, local)  
**Config:** seed=42, zero-overlap KB/test split

| Dataset | Test | KB | Source |
|---------|------|----|--------|
| TruthfulQA | 615Q | ~100Q | random split seed=42 |
| MMLU | 1596Q (28×57) | 228Q (4×57) | stratified split seed=42 |
| ARC-Challenge | 1172Q | 1418Q | official test vs train+val |

**Estimated runtime:** ~5-6 hours total (4-bit is faster than fp16)

In [1]:
# ── Cell 1: Environment check ─────────────────────────────────────────────────
import sys, torch, importlib

required = ['transformers','sentence_transformers','faiss','datasets',
            'rouge_score','mauve','sklearn','tqdm','bitsandbytes','accelerate']
missing = [p for p in required if not importlib.util.find_spec(p)]
if missing: print('MISSING:', missing)
else: print('All packages OK')

print(f'Python: {sys.version[:6]}')
print(f'PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')
else:
    print('WARNING: No GPU!')

All packages OK
Python: 3.10.2
PyTorch: 2.5.1+cu121
GPU: Quadro RTX 8000
VRAM: 47.5 GB


In [2]:
# ── Cell 2: Config ────────────────────────────────────────────────────────────
import os, sys

BASE_DIR     = '/home/user/RAG_Best_Practices/RAG_best_practices-main'
MODELS_DIR   = '/home/user/RAG_Best_Practices/models'
DATASETS_DIR = '/home/user/RAG_Best_Practices/datasets'
OUTPUT_DIR   = '/home/user/RAG_Best_Practices/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(BASE_DIR)
sys.path.insert(0, BASE_DIR)

# ── Dataset sizes (PAPER MODE)
N_TRUTHFULQA     = 615   # test → KB = ~100Q
MMLU_PER_SUBJECT = 28    # test = 1596Q → KB = 228Q
# ARC: uses official split → test=1172Q, KB=train(1119)+val(299)=1418Q
RANDOM_SEED      = 42
QUANT            = '4bit'  # paper-compatible

# ── Method params
ICL_PARAMS  = {'top_k_docs': 1, 'max_gen_tokens': 25, 'num_beams': 2}

CFR_PARAMS  = {'alpha': 0.5, 'beta': 0.3, 'gamma': 0.2, 'focus': 20}

CAFD_PARAMS = {'max_new_tokens': 15, 'analysis_max_tokens': 40}

SDCP_PARAMS = {
    'cert_threshold': 0.65,
    'alpha': 0.45, 'beta': 0.35, 'gamma': 0.20,
    'top_k_retrieve': 15, 'top_k_context': 4,
    'max_gen_tokens': 25, 'max_pos_tokens': 20, 'max_neg_tokens': 20,
}

CHOICE_LABELS = ['A', 'B', 'C', 'D']

print('Config OK — PAPER MODE (4-bit)')
print(f'TQA: {N_TRUTHFULQA}Q test | MMLU: {MMLU_PER_SUBJECT}×57={MMLU_PER_SUBJECT*57}Q | ARC: 1172Q test')

Config OK — PAPER MODE (4-bit)
TQA: 615Q test | MMLU: 28×57=1596Q | ARC: 1172Q test


In [3]:
# ── Cell 3: Load all 3 datasets ───────────────────────────────────────────────
import gc, numpy as np, pandas as pd
from datasets import load_from_disk

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 1. TruthfulQA
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print('Loading TruthfulQA...')
tqa_raw = load_from_disk(f'{DATASETS_DIR}/truthfulqa').to_pandas()
tqa_all = tqa_raw[['question','best_answer','correct_answers','incorrect_answers']].copy()
tqa_all['correct_answers']   = tqa_all['correct_answers'].apply(lambda x: x.tolist() if isinstance(x, np.ndarray) else [x])
tqa_all['incorrect_answers'] = tqa_all['incorrect_answers'].apply(lambda x: x.tolist() if isinstance(x, np.ndarray) else [x])
tqa_all['best_answer']       = tqa_all['best_answer'].apply(lambda x: [x] if x else [])
tqa_all = tqa_all[(tqa_all['correct_answers'].apply(len) > 1) &
                   (tqa_all['incorrect_answers'].apply(len) > 1)].reset_index(drop=True)
tqa_test_idx = tqa_all.sample(n=N_TRUTHFULQA, random_state=RANDOM_SEED).index
tqa     = tqa_all.loc[tqa_test_idx].reset_index(drop=True)
tqa_kb  = tqa_all.drop(tqa_test_idx).reset_index(drop=True)
del tqa_raw; gc.collect()
print(f'  test={len(tqa)}Q | KB={len(tqa_kb)}Q')

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 2. MMLU
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print('Loading MMLU...')
mmlu_raw = load_from_disk(f'{DATASETS_DIR}/mmlu').to_pandas()

def mmlu_to_unified(row):
    choices   = list(row['choices'])
    ans_idx   = int(row['answer'])
    correct   = choices[ans_idx]
    incorrect = [choices[i] for i in range(len(choices)) if i != ans_idx]
    formatted_q = (row['question'] + '\n' +
                   '\n'.join(f'{CHOICE_LABELS[i]}) {choices[i]}' for i in range(len(choices))))
    return pd.Series({'question': formatted_q, 'question_plain': row['question'],
                      'subject': row['subject'], 'best_answer': [correct],
                      'correct_answers': [correct], 'incorrect_answers': incorrect,
                      'answer_idx': ans_idx, 'choices': choices})

mmlu_test_parts, mmlu_kb_parts = [], []
for subject, group in mmlu_raw.groupby('subject'):
    group = group.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    n_test = min(MMLU_PER_SUBJECT, len(group))
    mmlu_test_parts.append(group.iloc[:n_test])
    if len(group) > n_test: mmlu_kb_parts.append(group.iloc[n_test:])
mmlu    = pd.concat(mmlu_test_parts).reset_index(drop=True).apply(mmlu_to_unified, axis=1)
mmlu_kb = pd.concat(mmlu_kb_parts).reset_index(drop=True).apply(mmlu_to_unified, axis=1)
del mmlu_raw; gc.collect()
print(f'  test={len(mmlu)}Q ({mmlu["subject"].nunique()} subjects) | KB={len(mmlu_kb)}Q')

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3. ARC-Challenge  (official test vs train+val — zero overlap by design)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print('Loading ARC-Challenge...')

def arc_to_unified(row):
    texts  = list(row['choices']['text'])
    labels = list(row['choices']['label'])
    key    = str(row['answerKey'])
    # answerKey can be 'A'-'D' or '1'-'4'
    if key.isdigit():
        ans_idx = int(key) - 1
    else:
        ans_idx = labels.index(key) if key in labels else 0
    ans_idx = min(ans_idx, len(texts)-1)
    correct   = texts[ans_idx]
    incorrect = [texts[i] for i in range(len(texts)) if i != ans_idx]
    formatted_q = (row['question'] + '\n' +
                   '\n'.join(f'{labels[i]}) {texts[i]}' for i in range(len(texts))))
    return pd.Series({'question': formatted_q, 'question_plain': row['question'],
                      'best_answer': [correct], 'correct_answers': [correct],
                      'incorrect_answers': incorrect, 'answer_idx': ans_idx,
                      'choices': texts, 'arc_id': row['id']})

arc_test_raw = load_from_disk(f'{DATASETS_DIR}/arc_challenge_test').to_pandas()
arc_trn_raw  = load_from_disk(f'{DATASETS_DIR}/arc_challenge_train').to_pandas()
arc_val_raw  = load_from_disk(f'{DATASETS_DIR}/arc_challenge_validation').to_pandas()
arc_kb_raw   = pd.concat([arc_trn_raw, arc_val_raw], ignore_index=True)

arc     = arc_test_raw.apply(arc_to_unified, axis=1).reset_index(drop=True)
arc_kb  = arc_kb_raw.apply(arc_to_unified, axis=1).reset_index(drop=True)
del arc_test_raw, arc_trn_raw, arc_val_raw, arc_kb_raw; gc.collect()
print(f'  test={len(arc)}Q | KB={len(arc_kb)}Q (train+val, zero overlap)')

print('\nAll datasets ready!')
print(f'  TQA : test={len(tqa)}  KB={len(tqa_kb)}')
print(f'  MMLU: test={len(mmlu)}  KB={len(mmlu_kb)}')
print(f'  ARC : test={len(arc)}  KB={len(arc_kb)}')

Loading TruthfulQA...
  test=615Q | KB=100Q
Loading MMLU...
  test=1596Q (57 subjects) | KB=228Q
Loading ARC-Challenge...
  test=1172Q | KB=1418Q (train+val, zero overlap)

All datasets ready!
  TQA : test=615  KB=100
  MMLU: test=1596  KB=228
  ARC : test=1172  KB=1418


In [4]:
# ── Cell 4: Load model (4-bit) ────────────────────────────────────────────────
import gc, torch, psutil
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

def show_mem():
    ram  = psutil.virtual_memory()
    vram = torch.cuda.memory_allocated()/1024**3 if torch.cuda.is_available() else 0
    print(f'  RAM {ram.used/1024**3:.1f}/{ram.total/1024**3:.1f}GB | VRAM {vram:.1f}GB')

gc.collect(); torch.cuda.empty_cache(); show_mem()

MODEL_PATH = f'{MODELS_DIR}/mistral-7b'
EMBED_PATH = f'{MODELS_DIR}/minilm'

print(f'Loading Mistral-7B ({QUANT})...')
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
llm = AutoModelForCausalLM.from_pretrained(MODEL_PATH, quantization_config=bnb, device_map='auto')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, padding_side='left')
tokenizer.pad_token = tokenizer.eos_token
show_mem()

print('Loading MiniLM...')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
embed_model = SentenceTransformer(EMBED_PATH).to(DEVICE)
show_mem()
print('Models ready!')

  RAM 10.7/47.0GB | VRAM 0.0GB
Loading Mistral-7B (4bit)...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  RAM 11.4/47.0GB | VRAM 4.2GB
Loading MiniLM...
  RAM 11.4/47.0GB | VRAM 4.3GB
Models ready!


In [5]:
# ── Cell 5: Core utilities ────────────────────────────────────────────────────
import faiss, numpy as np, pandas as pd, torch, gc
from rouge_score import rouge_scorer as rs_module
from sklearn.metrics.pairwise import cosine_similarity
import mauve as mauve_lib
from tqdm import tqdm

INST_S = '[INST]'
INST_E = '[/INST]'
SYS    = 'You are a truthful expert question-answering bot and should correctly and concisely answer the following question'

def generate(prompts, max_new_tokens=25, do_sample=False,
             temperature=1.0, top_p=0.9, num_beams=1):
    enc = tokenizer(prompts, return_tensors='pt', padding=True,
                    truncation=True, max_length=2048).to(DEVICE)
    input_len = enc['input_ids'].shape[1]
    kwargs = dict(input_ids=enc['input_ids'],
                  attention_mask=enc['attention_mask'],
                  max_new_tokens=max_new_tokens,
                  pad_token_id=tokenizer.eos_token_id)
    if do_sample:
        kwargs.update(do_sample=True, temperature=temperature, top_p=top_p)
    else:
        kwargs['num_beams'] = num_beams
    with torch.no_grad():
        out = llm.generate(**kwargs)
    return [tokenizer.decode(r[input_len:], skip_special_tokens=True).strip() or 'I have no comment'
            for r in out]

def get_token_probs(prompt, max_new_tokens=20):
    enc = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = llm.generate(**enc, max_new_tokens=max_new_tokens,
                           return_dict_in_generate=True, output_scores=True,
                           pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out.sequences[0][enc['input_ids'].shape[1]:],
                             skip_special_tokens=True).strip()
    cert = 0.0
    if out.scores:
        probs = torch.softmax(out.scores[0][0], dim=-1)
        top2  = torch.topk(probs, 2).values
        cert  = (top2[0] - top2[1]).item()
    return text, cert

def build_index(dataset):
    embs = embed_model.encode(dataset['question'].tolist(),
                               show_progress_bar=True, batch_size=64)
    embs = np.array(embs, dtype=np.float32)
    faiss.normalize_L2(embs)
    idx = faiss.IndexFlatIP(embs.shape[1])
    idx.add(embs)
    return idx

def retrieve_from_kb(query, faiss_idx, kb_dataset, k=1):
    q_emb = np.array(embed_model.encode([query], show_progress_bar=False), dtype=np.float32)
    faiss.normalize_L2(q_emb)
    _, idxs = faiss_idx.search(q_emb, k)
    return [kb_dataset.iloc[i] for i in idxs[0] if i < len(kb_dataset)]

def clean_response(resp):
    for stop in ['\nQuestion:','\nQ:','\n---','\nIncorrect','\nCorrect',
                 '\nVERIFIED','\nExample','\n\n','\nMy initial','\nContext:',
                 '\nFor the question', '\nCommon']:
        if stop in resp: resp = resp[:resp.index(stop)]
    return resp.strip().strip('"').strip("'") or 'I have no comment'

def compute_metrics(generated, references):
    scorer = rs_module.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
    r1s, r2s, rls, ecss = [], [], [], []
    for gen, refs in zip(generated, references):
        best_r1 = best_r2 = best_rl = 0
        for ref in refs:
            if not ref: continue
            s = scorer.score(ref, gen)
            best_r1 = max(best_r1, s['rouge1'].fmeasure)
            best_r2 = max(best_r2, s['rouge2'].fmeasure)
            best_rl = max(best_rl, s['rougeL'].fmeasure)
        r1s.append(best_r1*100); r2s.append(best_r2*100); rls.append(best_rl*100)
        try:
            embs = embed_model.encode([refs[0], gen])
            ecss.append(float(cosine_similarity([embs[0]], [embs[1]])[0][0])*100)
        except: ecss.append(0.0)
    return np.array(r1s), np.array(r2s), np.array(rls), np.array(ecss)

def compute_mauve(generated, references):
    try:
        refs  = [r[0] if isinstance(r, list) else r for r in references]
        valid = [(g, r) for g, r in zip(generated, refs) if g and r]
        if len(valid) < 10: return 0.0
        gens_v, refs_v = zip(*valid)
        result = mauve_lib.compute_mauve(p_text=list(refs_v), q_text=list(gens_v),
                                          device_id=0, max_text_length=256, verbose=False,
                                          featurize_model_name='gpt2')
        return result.mauve * 100
    except Exception as e:
        print(f'  MAUVE error: {e}'); return 0.0

def compute_accuracy(generated, dataset):
    correct = 0
    for gen, (_, row) in zip(generated, dataset.iterrows()):
        correct_text = row['best_answer'][0] if isinstance(row['best_answer'], list) else row['best_answer']
        ans_idx = int(row.get('answer_idx', -1))
        if ans_idx >= 0:
            label = CHOICE_LABELS[ans_idx]
            if label in gen[:5] or correct_text.lower() in gen.lower(): correct += 1
        else:
            if correct_text.lower() in gen.lower(): correct += 1
    return correct / len(generated) * 100

def pack_result(method, dataset_name, generated, references, dataset, prior_log=None):
    r1, r2, rl, ecs = compute_metrics(generated, references)
    mauve_score     = compute_mauve(generated, references)
    res = {'method': f'{method}-{dataset_name}', 'dataset': dataset_name,
           'R1': float(r1.mean()), 'R2': float(r2.mean()),
           'RL': float(rl.mean()), 'ECS': float(ecs.mean()),
           'MAUVE': mauve_score, 'generated': generated, 'references': references}
    if 'answer_idx' in dataset.columns:
        res['Accuracy'] = compute_accuracy(generated, dataset)
    else:
        res['Accuracy'] = 0.0
    if prior_log:
        scorer_p = rs_module.RougeScorer(['rouge1'], use_stemmer=True)
        pos_r1s  = [scorer_p.score(refs[0] if isinstance(refs,list) else refs, l['p_pos'])['rouge1'].fmeasure*100
                    for l, refs in zip(prior_log, references) if l.get('p_pos') and refs]
        res['pos_quality']   = float(np.mean(pos_r1s)) if pos_r1s else 0.0
        res['uncertain_pct'] = sum(1 for l in prior_log if l['mode']=='uncertain') / len(prior_log) * 100
        res['avg_cert']      = float(np.mean([l['cert'] for l in prior_log]))
        res['prior_log']     = prior_log
    acc_str = f" Acc={res['Accuracy']:.1f}%" if res['Accuracy'] > 0 else ''
    print(f'  R1={r1.mean():.2f} R2={r2.mean():.2f} RL={rl.mean():.2f} '
          f'ECS={ecs.mean():.2f} MAUVE={mauve_score:.2f}{acc_str}')
    return res

print('Utilities ready ✓')

Utilities ready ✓


In [6]:
# ── Cell 6: ICL1D+ ────────────────────────────────────────────────────────────
# Paper baseline: retrieve 1 similar question from KB,
# show its correct + incorrect answer as contrastive example.

def run_icl1d_plus(test_data, kb_data, dataset_name):
    print(f'\n=== ICL1D+ | {dataset_name} | test={len(test_data)}Q  KB={len(kb_data)}Q ===')
    faiss_idx = build_index(kb_data)
    generated, references = [], []

    for _, row in tqdm(test_data.iterrows(), total=len(test_data), desc='ICL1D+'):
        query       = row['question']
        best_answer = row['best_answer'] if isinstance(row['best_answer'], list) else [row['best_answer']]
        retrieved   = retrieve_from_kb(query, faiss_idx, kb_data, k=1)

        if retrieved:
            kb = retrieved[0]
            kb_q = kb.get('question_plain', kb['question'])
            kb_correct   = kb['best_answer'][0] if isinstance(kb['best_answer'], list) and kb['best_answer'] else str(kb['best_answer'])
            kb_incorrect = kb['incorrect_answers'][0] if isinstance(kb['incorrect_answers'], list) and kb['incorrect_answers'] else ''
            prompt = (f'{INST_S}{SYS}\n'
                      f'Q: {kb_q}\nCorrect answer: {kb_correct}\nIncorrect answer: {kb_incorrect}\n---\n'
                      f'Question: {query} Answer:{INST_E}')
        else:
            prompt = f'{INST_S}{SYS}\nQuestion: {query}\nAnswer:{INST_E}'

        resp = generate([prompt], max_new_tokens=ICL_PARAMS['max_gen_tokens'],
                        num_beams=ICL_PARAMS['num_beams'])
        generated.append(clean_response(resp[0]))
        references.append(best_answer)

    return pack_result('ICL1D+', dataset_name, generated, references, test_data)

print('ICL1D+ ready ✓')

ICL1D+ ready ✓


In [7]:
# ── Cell 7: CFR (Contrastive Focus Reranker) ──────────────────────────────────
# CFR(s) = α·sim(s,q) + β·sim(s,kb_correct) - γ·sim(s,kb_incorrect)
import re

def run_cfr(test_data, kb_data, dataset_name):
    print(f'\n=== CFR | {dataset_name} | test={len(test_data)}Q  KB={len(kb_data)}Q ===')
    alpha = CFR_PARAMS['alpha']
    beta  = CFR_PARAMS['beta']
    gamma = CFR_PARAMS['gamma']
    focus = CFR_PARAMS['focus']
    faiss_idx = build_index(kb_data)
    generated, references = [], []

    for idx, row in tqdm(test_data.iterrows(), total=len(test_data), desc='CFR'):
        query       = row['question']
        best_answer = row['best_answer'] if isinstance(row['best_answer'], list) else [row['best_answer']]
        retrieved   = retrieve_from_kb(query, faiss_idx, kb_data, k=focus)

        if retrieved:
            kb          = retrieved[0]
            kb_correct  = kb['best_answer'][0] if isinstance(kb['best_answer'], list) and kb['best_answer'] else str(kb['best_answer'])
            kb_incorrect= kb['incorrect_answers'][0] if isinstance(kb['incorrect_answers'], list) and kb['incorrect_answers'] else ''
            kb_q        = kb.get('question_plain', kb['question'])

            sentences = []
            for doc in retrieved:
                for s in re.split(r'(?<=[.!?])\s+', doc['question']):
                    if len(s.strip()) > 10: sentences.append(s.strip())

            if sentences and kb_correct and kb_incorrect:
                s_embs = embed_model.encode(sentences, show_progress_bar=False)
                q_emb  = embed_model.encode([query],        show_progress_bar=False)
                c_emb  = embed_model.encode([kb_correct],   show_progress_bar=False)
                i_emb  = embed_model.encode([kb_incorrect], show_progress_bar=False)
                scores = (alpha * cosine_similarity(s_embs, q_emb).flatten() +
                          beta  * cosine_similarity(s_embs, c_emb).flatten() -
                          gamma * cosine_similarity(s_embs, i_emb).flatten())
                context = ' '.join([sentences[i] for i in np.argsort(-scores)[:5]])
                prompt  = (f'{INST_S}{SYS}\nContext: {context}\n'
                           f'Example — Q: {kb_q}\nCorrect: {kb_correct}\nIncorrect: {kb_incorrect}\n'
                           f'Question: {query}\nAnswer:{INST_E}')
            else:
                prompt = (f'{INST_S}{SYS}\nExample — Q: {kb_q}\n'
                          f'Question: {query}\nAnswer:{INST_E}')
        else:
            prompt = f'{INST_S}{SYS}\nQuestion: {query}\nAnswer:{INST_E}'

        generated.append(clean_response(generate([prompt], max_new_tokens=25, num_beams=2)[0]))
        references.append(best_answer)
        if idx % 50 == 0: gc.collect(); torch.cuda.empty_cache()

    return pack_result('CFR', dataset_name, generated, references, test_data)

print('CFR ready ✓')

CFR ready ✓


In [8]:
# ── Cell 8: CAFD-LC (Contrastive Atomic Fact Decomposition) ───────────────────

def precompute_analyses(kb_data):
    print(f'  Pre-computing fact analyses on {len(kb_data)}Q KB...')
    cache = {}
    for idx, row in tqdm(kb_data.iterrows(), total=len(kb_data), desc='CAFD precompute'):
        correct   = row['best_answer'][0] if isinstance(row['best_answer'], list) and row['best_answer'] else str(row['best_answer'])
        incorrect = row['incorrect_answers'][0] if row['incorrect_answers'] else ''
        question  = row.get('question_plain', row['question'])
        if not correct or not incorrect:
            cache[idx] = None; continue
        prompt = (f'{INST_S}For the question: "{question}"\n'
                  f'Correct answer: "{correct}"\n'
                  f'Wrong answer: "{incorrect}"\n'
                  f'In one sentence explain why correct is right and wrong is misleading:{INST_E}')
        analysis = generate([prompt], max_new_tokens=CAFD_PARAMS['analysis_max_tokens'], num_beams=2)[0]
        for stop in ['\n\n','\nQuestion:','\nFor the question','\n---']:
            if stop in analysis: analysis = analysis[:analysis.index(stop)]
        cache[idx] = {'correct': correct, 'incorrect': incorrect,
                      'analysis': analysis.strip().strip('"')[:200],
                      'question': question}
        if idx % 50 == 0: gc.collect(); torch.cuda.empty_cache()
    print(f'  Cached {sum(1 for v in cache.values() if v)} analyses')
    return cache

def run_cafd_lc(test_data, kb_data, dataset_name, cache=None):
    print(f'\n=== CAFD-LC | {dataset_name} | test={len(test_data)}Q  KB={len(kb_data)}Q ===')
    faiss_idx = build_index(kb_data)
    if cache is None:
        cache = precompute_analyses(kb_data)
    generated, references = [], []

    for idx, row in tqdm(test_data.iterrows(), total=len(test_data), desc='CAFD-LC'):
        query       = row['question']
        best_answer = row['best_answer'] if isinstance(row['best_answer'], list) else [row['best_answer']]
        retrieved   = retrieve_from_kb(query, faiss_idx, kb_data, k=1)

        if retrieved:
            kb    = retrieved[0]
            fact  = cache.get(kb.name)
            if fact:
                kb_q = fact.get('question', kb['question'])
                ctx  = (f'Factual distinction example:\n'
                        f'- Q: {kb_q}\n- FACT: {fact["correct"]}\n- MYTH: {fact["incorrect"]}'
                        + (f'\n- WHY: {fact["analysis"]}' if fact.get('analysis') else ''))
                prompt = (f'{INST_S}{SYS}\n{ctx}\n---\n'
                          f'Question: {query}\nAnswer:{INST_E}')
            else:
                kb_q = kb.get('question_plain', kb['question'])
                prompt = (f'{INST_S}{SYS}\nExample: Q: {kb_q}\n'
                          f'Question: {query}\nAnswer:{INST_E}')
        else:
            prompt = f'{INST_S}{SYS}\nQuestion: {query}\nAnswer:{INST_E}'

        generated.append(clean_response(generate([prompt], max_new_tokens=CAFD_PARAMS['max_new_tokens'], num_beams=2)[0]))
        references.append(best_answer)
        if idx % 50 == 0: gc.collect(); torch.cuda.empty_cache()

    return pack_result('CAFD-LC', dataset_name, generated, references, test_data), cache

print('CAFD-LC ready ✓')

CAFD-LC ready ✓


In [9]:
# ── Cell 9: SDCP (Self-Distilled Contrastive Priors) ─────────────────────────
# FULLY label-free: P_pos and P_neg generated by the model itself.

def generate_sdcp_priors(query, dataset_name, params):
    pos_prompt = f'{INST_S}{SYS}\nQuestion: {query}\nAnswer concisely:{INST_E}'
    p_pos, cert = get_token_probs(pos_prompt, max_new_tokens=params['max_pos_tokens'])
    p_pos = clean_response(p_pos)

    q_plain = query.split('\n')[0] if '\n' in query else query
    if dataset_name in ['MMLU', 'ARC']:
        neg_prompt = (f'{INST_S}What is a plausible but INCORRECT answer that students '
                      f'commonly give for this type of question?\n'
                      f'Question: {q_plain}\nCommon wrong answer (very short):{INST_E}')
    else:
        neg_prompt = (f'{INST_S}What is a common misconception or false belief '
                      f'that people hold about this topic?\n'
                      f'Question: {q_plain}\nCommon wrong belief (very short):{INST_E}')

    p_neg = clean_response(generate([neg_prompt], max_new_tokens=params['max_neg_tokens'], num_beams=1)[0])
    return p_pos, p_neg, cert

def run_sdcp(test_data, kb_data, dataset_name, params=None):
    if params is None: params = SDCP_PARAMS
    print(f'\n=== SDCP | {dataset_name} | test={len(test_data)}Q  KB={len(kb_data)}Q ===')
    print(f'    α={params["alpha"]} β={params["beta"]} γ={params["gamma"]} cert_θ={params["cert_threshold"]}')
    faiss_idx = build_index(kb_data)
    generated, references, prior_log = [], [], []

    for idx, row in tqdm(test_data.iterrows(), total=len(test_data), desc='SDCP'):
        query       = row['question']
        best_answer = row['best_answer'] if isinstance(row['best_answer'], list) else [row['best_answer']]

        # Step 1: self-distill priors
        p_pos, p_neg, cert = generate_sdcp_priors(query, dataset_name, params)

        # Step 2: prior-guided contrastive retrieval
        retrieved = retrieve_from_kb(query, faiss_idx, kb_data, k=params['top_k_retrieve'])

        prompt = None
        if retrieved and p_pos:
            sentences = []
            for doc in retrieved:
                sentences.append(doc['question'])
                ba = doc['best_answer']
                ba_t = ba[0] if isinstance(ba, list) and ba else str(ba)
                if ba_t and len(ba_t) > 3: sentences.append(ba_t)
            sentences = [s for s in sentences if s and len(s.strip()) > 5]

            if sentences:
                s_embs  = embed_model.encode(sentences, show_progress_bar=False)
                q_emb   = embed_model.encode([query],  show_progress_bar=False)
                pos_emb = embed_model.encode([p_pos],  show_progress_bar=False)
                neg_emb = embed_model.encode([p_neg],  show_progress_bar=False) if p_neg else None
                q_sims  = cosine_similarity(s_embs, q_emb).flatten()
                p_sims  = cosine_similarity(s_embs, pos_emb).flatten()
                n_sims  = (cosine_similarity(s_embs, neg_emb).flatten()
                           if neg_emb is not None else np.zeros(len(sentences)))
                sdcp_sc = params['alpha']*q_sims + params['beta']*p_sims - params['gamma']*n_sims
                context = ' '.join([sentences[i] for i in np.argsort(-sdcp_sc)[:params['top_k_context']]])

                kb_ex       = retrieved[0]
                kb_correct  = kb_ex['best_answer'][0] if isinstance(kb_ex['best_answer'], list) and kb_ex['best_answer'] else str(kb_ex['best_answer'])
                kb_incorrect= kb_ex['incorrect_answers'][0] if isinstance(kb_ex['incorrect_answers'], list) and kb_ex['incorrect_answers'] else ''
                kb_q        = kb_ex.get('question_plain', kb_ex['question'])

                # Step 3: certainty-adaptive prompt
                if cert < params['cert_threshold']:
                    prompt = (f'{INST_S}{SYS}\nRetrieved context: {context}\n'
                              f'Example — Q: {kb_q}\n  Correct: {kb_correct}\n  Incorrect: {kb_incorrect}\n'
                              f'My initial thought: {p_pos}\nCommon mistake to avoid: {p_neg}\n'
                              f'Question: {query}\nVerified answer:{INST_E}')
                else:
                    prompt = (f'{INST_S}{SYS}\nContext: {context}\n'
                              f'Initial answer: {p_pos}\nCommon mistake to avoid: {p_neg}\n'
                              f'Question: {query}\nRefined answer:{INST_E}')

        if prompt is None:
            prompt = f'{INST_S}{SYS}\nQuestion: {query}\nAnswer:{INST_E}'

        # Step 4: final generation
        resp  = generate([prompt], max_new_tokens=params['max_gen_tokens'], num_beams=2)
        final = clean_response(resp[0])
        generated.append(final)
        references.append(best_answer)
        prior_log.append({'p_pos': p_pos, 'p_neg': p_neg, 'cert': cert,
                          'mode': 'uncertain' if cert < params['cert_threshold'] else 'confident'})
        if idx % 30 == 0: gc.collect(); torch.cuda.empty_cache()

    uncertain_n = sum(1 for l in prior_log if l['mode']=='uncertain')
    print(f'  uncertain={uncertain_n}/{len(test_data)} ({uncertain_n/len(test_data)*100:.0f}%) '
          f'avg_cert={np.mean([l["cert"] for l in prior_log]):.3f}')
    return pack_result('SDCP', dataset_name, generated, references, test_data, prior_log)

print('SDCP ready ✓')

SDCP ready ✓


In [10]:
# ── Cell 10: RUN — TruthfulQA ─────────────────────────────────────────────────
import time
all_results = {}

print('=' * 65)
print(f'PHASE 1: TruthfulQA  (test={len(tqa)}Q | KB={len(tqa_kb)}Q)')
print('=' * 65)

t0 = time.time()
all_results['ICL1D_TQA']   = run_icl1d_plus(tqa, tqa_kb, 'TQA')
print(f'  ✓ ICL1D+  {(time.time()-t0)/60:.1f} min'); t0 = time.time()

all_results['CFR_TQA']     = run_cfr(tqa, tqa_kb, 'TQA')
print(f'  ✓ CFR     {(time.time()-t0)/60:.1f} min'); t0 = time.time()

all_results['CAFD_TQA'], tqa_cache = run_cafd_lc(tqa, tqa_kb, 'TQA')
print(f'  ✓ CAFD-LC {(time.time()-t0)/60:.1f} min'); t0 = time.time()

all_results['SDCP_TQA']    = run_sdcp(tqa, tqa_kb, 'TQA')
print(f'  ✓ SDCP    {(time.time()-t0)/60:.1f} min')

print('\nTruthfulQA complete!')

PHASE 1: TruthfulQA  (test=615Q | KB=100Q)

=== ICL1D+ | TQA | test=615Q  KB=100Q ===


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

ICL1D+: 100%|██████████| 615/615 [19:00<00:00,  1.86s/it]


Featurizing p:   0%|          | 0/615 [00:00<?, ?it/s]

Featurizing q:   0%|          | 0/615 [00:00<?, ?it/s]

...

  uncertain=241/615 (39%) avg_cert=0.688


Featurizing p:   0%|          | 0/615 [00:00<?, ?it/s]

Featurizing q:   0%|          | 0/615 [00:00<?, ?it/s]

WARNING clustering 1230 points to 62 centroids: please provide at least 2418 training points


  R1=34.26 R2=18.47 RL=30.52 ECS=64.92 MAUVE=9.54
  ✓ SDCP    42.3 min

TruthfulQA complete!


In [11]:
# ── Cell 11: RUN — MMLU ───────────────────────────────────────────────────────
print('=' * 65)
print(f'PHASE 2: MMLU  (test={len(mmlu)}Q | KB={len(mmlu_kb)}Q)')
print('=' * 65)

t0 = time.time()
all_results['ICL1D_MMLU']   = run_icl1d_plus(mmlu, mmlu_kb, 'MMLU')
print(f'  ✓ ICL1D+  {(time.time()-t0)/60:.1f} min'); t0 = time.time()

all_results['CFR_MMLU']     = run_cfr(mmlu, mmlu_kb, 'MMLU')
print(f'  ✓ CFR     {(time.time()-t0)/60:.1f} min'); t0 = time.time()

all_results['CAFD_MMLU'], mmlu_cache = run_cafd_lc(mmlu, mmlu_kb, 'MMLU')
print(f'  ✓ CAFD-LC {(time.time()-t0)/60:.1f} min'); t0 = time.time()

all_results['SDCP_MMLU']    = run_sdcp(mmlu, mmlu_kb, 'MMLU')
print(f'  ✓ SDCP    {(time.time()-t0)/60:.1f} min')

print('\nMMELU complete!')

PHASE 2: MMLU  (test=1596Q | KB=228Q)

=== ICL1D+ | MMLU | test=1596Q  KB=228Q ===


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

ICL1D+: 100%|██████████| 1596/1596 [50:59<00:00,  1.92s/it]


Featurizing p:   0%|          | 0/1596 [00:00<?, ?it/s]

Featurizing q:   0%|          | 0/1596 [00:00<?, ?it/s]

...

  uncertain=655/1596 (41%) avg_cert=0.663


Featurizing p:   0%|          | 0/1596 [00:00<?, ?it/s]

Featurizing q:   0%|          | 0/1596 [00:00<?, ?it/s]

WARNING clustering 3192 points to 160 centroids: please provide at least 6240 training points


  R1=32.34 R2=23.47 RL=31.33 ECS=48.14 MAUVE=47.30 Acc=45.2%
  ✓ SDCP    114.2 min

MMELU complete!


In [12]:
# ── Cell 12: RUN — ARC-Challenge ─────────────────────────────────────────────
print('=' * 65)
print(f'PHASE 3: ARC-Challenge  (test={len(arc)}Q | KB={len(arc_kb)}Q)')
print('=' * 65)

t0 = time.time()
all_results['ICL1D_ARC']   = run_icl1d_plus(arc, arc_kb, 'ARC')
print(f'  ✓ ICL1D+  {(time.time()-t0)/60:.1f} min'); t0 = time.time()

all_results['CFR_ARC']     = run_cfr(arc, arc_kb, 'ARC')
print(f'  ✓ CFR     {(time.time()-t0)/60:.1f} min'); t0 = time.time()

all_results['CAFD_ARC'], arc_cache = run_cafd_lc(arc, arc_kb, 'ARC')
print(f'  ✓ CAFD-LC {(time.time()-t0)/60:.1f} min'); t0 = time.time()

all_results['SDCP_ARC']    = run_sdcp(arc, arc_kb, 'ARC')
print(f'  ✓ SDCP    {(time.time()-t0)/60:.1f} min')

print('\nARC-Challenge complete!')
print('\n ALL 3 DATASETS DONE!')

PHASE 3: ARC-Challenge  (test=1172Q | KB=1418Q)

=== ICL1D+ | ARC | test=1172Q  KB=1418Q ===


Batches:   0%|          | 0/23 [00:00<?, ?it/s]

ICL1D+: 100%|██████████| 1172/1172 [37:41<00:00,  1.93s/it]


Featurizing p:   0%|          | 0/1172 [00:00<?, ?it/s]

Featurizing q:   0%|          | 0/1172 [00:00<?, ?it/s]

...

  uncertain=470/1172 (40%) avg_cert=0.673


Featurizing p:   0%|          | 0/1172 [00:00<?, ?it/s]

Featurizing q:   0%|          | 0/1172 [00:00<?, ?it/s]

WARNING clustering 2344 points to 117 centroids: please provide at least 4563 training points


  R1=35.02 R2=26.89 RL=34.36 ECS=53.15 MAUVE=23.36 Acc=56.1%
  ✓ SDCP    81.4 min

ARC-Challenge complete!

 ALL 3 DATASETS DONE!


In [13]:
# ── Cell 13: Final Comparison Table ──────────────────────────────────────────

PAPER_BASELINES = {
    'TQA':  {'Base (paper)':   {'R1':26.81,'R2':13.26,'RL':23.86,'ECS':56.44,'MAUVE':72.92},
             'ICL1D+ (paper)': {'R1':30.62,'R2':17.45,'RL':27.79,'ECS':58.96,'MAUVE':73.86}},
    'MMLU': {'Base (paper)':   {'R1':10.42,'R2': 1.90,'RL': 8.91,'ECS':29.41,'MAUVE':40.51},
             'ICL1D+ (paper)': {'R1':26.01,'R2':17.46,'RL':24.90,'ECS':47.04,'MAUVE':37.24}},
    'ARC':  {}  # no prior baselines for ARC
}

OUR_METHOD_KEYS = [
    ('ICL1D+',  ['ICL1D_TQA',  'ICL1D_MMLU',  'ICL1D_ARC']),
    ('CFR',     ['CFR_TQA',    'CFR_MMLU',    'CFR_ARC']),
    ('CAFD-LC', ['CAFD_TQA',   'CAFD_MMLU',   'CAFD_ARC']),
    ('SDCP',    ['SDCP_TQA',   'SDCP_MMLU',   'SDCP_ARC']),
]

def print_table(dataset_name, key):
    is_mc = dataset_name in ['MMLU', 'ARC']
    print(f'\n{"+" + "="*75 + "+"}')
    print(f'| {dataset_name:<73}|')
    print(f'{"+" + "="*75 + "+"}')
    hdr = f'{"Method":<22} {"R1":>6} {"R2":>6} {"RL":>6} {"ECS":>6} {"MAUVE":>7}'
    if is_mc: hdr += f' {"Acc":>7}'
    print(hdr)
    print('-'*75)
    for name, vals in PAPER_BASELINES.get(dataset_name, {}).items():
        line = f'{name:<22} {vals["R1"]:>6.2f} {vals["R2"]:>6.2f} {vals["RL"]:>6.2f} {vals["ECS"]:>6.2f} {vals["MAUVE"]:>7.2f}'
        if is_mc: line += '       —'
        print(line)
    if PAPER_BASELINES.get(dataset_name): print('─'*75)
    for method_name, keys in OUR_METHOD_KEYS:
        rkey = keys[['TQA','MMLU','ARC'].index(dataset_name)]
        if rkey not in all_results: continue
        r = all_results[rkey]
        label_flag = '✗' if method_name == 'SDCP' else '✓'
        line = (f'{method_name+" (ours,"+label_flag+")":<22} '
                f'{r["R1"]:>6.2f} {r["R2"]:>6.2f} {r["RL"]:>6.2f} '
                f'{r["ECS"]:>6.2f} {r["MAUVE"]:>7.2f}')
        if is_mc: line += f' {r.get("Accuracy",0):>6.1f}%'
        print(line)

for ds in ['TQA','MMLU','ARC']:
    print_table(ds, ds)

# SDCP analysis summary
print('\n' + '='*65)
print('SDCP Prior Analysis')
print('='*65)
print(f'{"Dataset":<8} {"uncertain%":>11} {"avg_cert":>10} {"pos_quality":>12}')
print('-'*45)
for ds, key in [('TQA','SDCP_TQA'),('MMLU','SDCP_MMLU'),('ARC','SDCP_ARC')]:
    if key in all_results:
        r = all_results[key]
        print(f'{ds:<8} {r.get("uncertain_pct",0):>10.1f}% {r.get("avg_cert",0):>10.3f} {r.get("pos_quality",0):>11.2f}')


+===========================================================================+
| TQA                                                                      |
+===========================================================================+
Method                     R1     R2     RL    ECS   MAUVE
---------------------------------------------------------------------------
Base (paper)            26.81  13.26  23.86  56.44   72.92
ICL1D+ (paper)          30.62  17.45  27.79  58.96   73.86
───────────────────────────────────────────────────────────────────────────
ICL1D+ (ours,✓)         35.82  20.85  32.31  65.42   13.59
CFR (ours,✓)            34.74  18.88  30.90  64.33   11.18
CAFD-LC (ours,✓)        40.32  24.74  37.51  63.82   14.23
SDCP (ours,✗)           34.26  18.47  30.52  64.92    9.54

+===========================================================================+
| MMLU                                                                     |
+============================================

In [14]:
# ── Cell 14: Save all results ─────────────────────────────────────────────────
from datetime import datetime
import json

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
summary = {}
for key, r in all_results.items():
    summary[key] = {
        'method': r['method'], 'dataset': r['dataset'],
        'R1':  round(r['R1'],  3), 'R2':  round(r['R2'],  3),
        'RL':  round(r['RL'],  3), 'ECS': round(r['ECS'], 3),
        'MAUVE':    round(r['MAUVE'], 3),
        'Accuracy': round(r.get('Accuracy', 0), 2),
    }
    if 'pos_quality' in r:
        summary[key].update({
            'pos_quality':   round(r['pos_quality'], 3),
            'uncertain_pct': round(r['uncertain_pct'], 1),
            'avg_cert':      round(r['avg_cert'], 4),
        })

out_path = f'{OUTPUT_DIR}/full_4bit_summary_{ts}.json'
with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2)

# Save per-method CSVs
for key, r in all_results.items():
    df = pd.DataFrame({
        'generated': r['generated'],
        'reference': [x[0] if isinstance(x, list) else x for x in r['references']],
    })
    df.to_csv(f'{OUTPUT_DIR}/{key}_full4bit_{ts}.csv', index=False, quoting=1)

print(f'Results saved → {out_path}')
print('\nSummary:')
print(json.dumps(summary, indent=2))

Results saved → /home/user/RAG_Best_Practices/outputs/full_4bit_summary_20260429_210909.json

Summary:
{
  "ICL1D_TQA": {
    "method": "ICL1D+-TQA",
    "dataset": "TQA",
    "R1": 35.821,
    "R2": 20.852,
    "RL": 32.314,
    "ECS": 65.423,
    "MAUVE": 13.588,
    "Accuracy": 0.0
  },
  "CFR_TQA": {
    "method": "CFR-TQA",
    "dataset": "TQA",
    "R1": 34.744,
    "R2": 18.88,
    "RL": 30.896,
    "ECS": 64.333,
    "MAUVE": 11.184,
    "Accuracy": 0.0
  },
  "CAFD_TQA": {
    "method": "CAFD-LC-TQA",
    "dataset": "TQA",
    "R1": 40.322,
    "R2": 24.74,
    "RL": 37.507,
    "ECS": 63.818,
    "MAUVE": 14.226,
    "Accuracy": 0.0
  },
  "SDCP_TQA": {
    "method": "SDCP-TQA",
    "dataset": "TQA",
    "R1": 34.263,
    "R2": 18.466,
    "RL": 30.516,
    "ECS": 64.916,
    "MAUVE": 9.537,
    "Accuracy": 0.0,
    "pos_quality": 33.21,
    "uncertain_pct": 39.2,
    "avg_cert": 0.6885
  },
  "ICL1D_MMLU": {
    "method": "ICL1D+-MMLU",
    "dataset": "MMLU",
    "R1": 39.88